In [68]:
from ase.build import graphene
from mace.calculators import MACECalculator
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter
import numpy as np

from ase.visualize import view
import os

In [69]:
model = MACECalculator("../../MACE.model", device = "cuda")

Using head Default out of ['Default']
No dtype selected, switching to float32 to match model dtype.


/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:197: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/gabrielecolombo/Thesis/PROJECT/.venv/lib/python3.12/site-packages/torch/jit/_serialization.py:176: DeprecationWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


In [70]:
primitive = graphene()
primitive.cell[2][2] = 50
primitive.calc = model

filter = FrechetCellFilter(primitive, [1, 1, 0, 0, 0, 0])
opt = BFGS(filter)
opt.run(fmax = 1e-5)

      Step     Time          Energy          fmax
BFGS:    0 15:57:26      -15.917215        0.003647
BFGS:    1 15:57:26      -15.917213        0.002278
BFGS:    2 15:57:26      -15.917213        0.000038
BFGS:    3 15:57:26      -15.917213        0.000014
BFGS:    4 15:57:26      -15.917215        0.000002


np.True_

In [71]:
versor = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]])
strain = 0.178 * versor
deformation = np.eye(3) + strain

cell0 = primitive.cell.copy()

In [72]:
from ase.build import make_supercell

In [73]:
repeated = make_supercell(primitive, np.array([[26, 26, 0], [-8, 8, 0], [0, 0, 1]]))
repeated.rotate(repeated.cell[0], 'x', rotate_cell=True)
cell = repeated.cell @ deformation
repeated.set_cell(cell, scale_atoms=True)
repeated.calc = model

amplitude = 2e-2

displacements = np.random.normal(0, amplitude, size = (len(repeated), 3))
displacements[:,2] = 0
repeated.positions += displacements

In [74]:
repeated.cell

Cell([[63.955553645551404, 0.0, 0.0], [1.2898099939206986e-06, 40.15141084734969, 0.0], [0.0, 0.0, 50.0]])

In [75]:
view(repeated, viewer="ngl")

In [76]:
OUTDIR = "ARMCHAIRRELAX"
os.makedirs(OUTDIR, exist_ok=True)

In [77]:
opt = BFGS(repeated, trajectory=OUTDIR + "/armchair.traj")
opt.run(fmax = 1e-4, steps=500)

      Step     Time          Energy          fmax
BFGS:    0 15:57:27    -6034.553223        3.738523
BFGS:    1 15:57:28    -6053.815918        1.378898
BFGS:    2 15:57:29    -6057.323242        0.421081
BFGS:    3 15:57:29    -6057.552734        0.269015
BFGS:    4 15:57:30    -6057.700684        0.215843
BFGS:    5 15:57:31    -6057.761719        0.141909
BFGS:    6 15:57:32    -6057.801270        0.096566
BFGS:    7 15:57:33    -6057.813477        0.067210
BFGS:    8 15:57:33    -6057.833008        0.056755
BFGS:    9 15:57:34    -6057.840820        0.047200
BFGS:   10 15:57:35    -6057.850098        0.037659
BFGS:   11 15:57:36    -6057.849609        0.033424
BFGS:   12 15:57:36    -6057.851074        0.024469
BFGS:   13 15:57:37    -6057.853516        0.019413
BFGS:   14 15:57:38    -6057.854980        0.016500
BFGS:   15 15:57:39    -6057.852539        0.016483
BFGS:   16 15:57:40    -6057.852539        0.014833
BFGS:   17 15:57:41    -6057.857422        0.013649
BFGS:   18 15:

np.False_